In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

In [6]:
# Load your data
data = pd.read_csv("data.csv")

# Data cleaning and preprocessing
def extract_numeric(value):
    if pd.isna(value):
        return np.nan
    values = [float(num) for num in value.replace(",", "").split() if num.replace('.', '', 1).isdigit()]
    return np.mean(values) if values else np.nan

In [8]:
def calculate_conversion_rate(C_initial, C_final):
    """
    Calculate the conversion rate of a reactant.

    Parameters:
        C_initial (float): Initial concentration or moles of the reactant.
        C_final (float): Final concentration or moles of the reactant.

    Returns:
        float: Conversion rate as a percentage.
    """
    if C_initial == 0:
        raise ValueError("Initial concentration cannot be zero.")
    
    conversion_rate = ((C_initial - C_final) / C_initial) * 100
    return conversion_rate

In [9]:
data['Conversion Rate'] = data.apply(
    lambda row: calculate_conversion_rate(row['Reactant_Concentration_Initial'], row['Reactant_Concentration_Final']),
    axis=1
)

In [13]:
data = data.drop(['Reactant_Concentration_Initial', 'Reactant_Concentration_Final'], axis=1)


In [14]:
print("Processed Dataset:")
print(data.head())

Processed Dataset:
   paper year                                        link  \
0      2022.0  https://doi.org/10.1016/j.fuel.2022.123249   
1      2022.0  https://doi.org/10.1016/j.fuel.2022.125548   
2      2022.0   https://doi.org/10.1016/j.ces.2022.117554   
3      2023.0   https://doi.org/10.1016/j.ces.2023.119654   
4      2021.0  https://doi.org/10.1016/j.fuel.2021.121818   

                 catalyst used and its specification  \
0  carbon-supported iron carbide, specifically Fe...   
1  NiRu/Al₂O₃, which is a bimetallic catalyst com...   
2   Ru/AC (Ruthenium supported on activated carbon).   
3  bimetallic catalyst  NiCu/C. synthesized throu...   
4  1. Palladium (Pd)\nSupported on activated bioc...   

                preparation temperature for catalyst  temperature(process)  \
0                       (500 °C, 700 °C, and 900 °C)                 280.0   
1                                             200 °C                 220.0   
2                                      not 

In [15]:
X = data.drop('Conversion Rate', axis=1)  # Features
y = data['Conversion Rate']  # Target

In [46]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



In [47]:
X_train = X_train.apply(pd.to_numeric, errors='coerce')
y_train = pd.to_numeric(y_train, errors='coerce')
X_train = X_train.dropna()
y_train = y_train[X_train.index]  # Align y_train with X_train


In [50]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

# Assuming data is preprocessed and cleaned
# Ensure correct shape and type
X_train = X_train.apply(pd.to_numeric, errors='coerce')
y_train = pd.to_numeric(y_train, errors='coerce')

# Handle missing values
X_train = X_train.fillna(X_train.mean())  # Filling missing values for features
y_train = y_train.fillna(y_train.mean())  # Filling missing values for target

# Check for shape and type
print(X_train.shape)  # Should be (n_samples, n_features)
print(y_train.shape)  # Should be (n_samples,)

# Convert to numpy if needed
X_train = X_train.values
y_train = y_train.values




(0, 13)
(0,)


In [51]:
# Train the Random Forest model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

ValueError: Found array with 0 sample(s) (shape=(0, 13)) while a minimum of 1 is required by RandomForestRegressor.

In [30]:
best_catalyst_custom = cleaned_data.loc[cleaned_data['predicted_yield_custom'].idxmax()]
print("Best Catalyst for Custom Conditions:")
print("Catalyst:", best_catalyst_custom['catalyst used and its specification'])
print("Predicted Yield (%):", best_catalyst_custom['predicted_yield_custom'])
print("Custom Temperature (°C):", custom_temperature)
print("Custom Pressure (MPa):", custom_pressure)


Best Catalyst for Custom Conditions:
Catalyst: carbon-supported iron carbide, specifically Fe-Fe3C/C . These catalysts were synthesized using a high-temperature solid-phase method under a reducing atmosphere (10% H2 / 90% Ar) Structural Composition: Characterization techniques such as X-ray diffraction (XRD) and Raman spectroscopy revealed that the catalysts consist of graphitized and amorphous carbon encapsulating iron carbide nanocrystals and metal iron nanoclusters Performance in Lignin Conversion: The Fe-Fe3C/C catalysts demonstrated high selectivity for aromatic monomers, achieving an 86 wt% selectivity for guaiacol and its alkyl derivatives at a 48 wt% yield during optimal experimental conditions Mechanism of Action: The catalysts function by utilizing acid sites to adsorb and break C-O bonds, while metal Fe acts as a hydrogen source to promote hydrogenation processes. This dual functionality enhances the overall efficiency of lignin depolymerization 

Predicted Yield (%): 4.0
Cu